In [ ]:
%load_ext autoreload
%autoreload 2

import os
import zarr
import polars as pl
from anngeno import AnnGeno

from scripts import get_burdens_chunky, get_burdens

import multiprocessing

num_cores = multiprocessing.cpu_count()
print(num_cores)

In [ ]:
anngeno_path = "/home/dnanexus/data_dir/anngeno_training.ag"
ag = AnnGeno(anngeno_path,  filemode="r", low_mem=True)
ag

In [ ]:
r = ag.get_region('ENSG00000123739')
r['annotations']

In [ ]:
import matplotlib.pyplot as plt

# Get non-null elements
absplice_dna_max = r['annotations']['AbSplice2_max']
# Plot histogram
plt.hist(absplice_dna_max.to_numpy(), bins=30, log=True)

In [ ]:
%%time

config_path = "/home/dnanexus/ukbgym/config_wgs_absplice2.yaml"
associations_df_path = '/home/dnanexus/data_dir/absplice2_assocs.parquet'
output_dir = "/home/dnanexus/debug_burdens"

adf = pl.read_parquet(associations_df_path).head(1)

get_burdens_chunky.compute_and_store_burdens(
    config_path=config_path,
    associations_df=adf,
    output_dir=output_dir,
    only_snps=True,
    na_mask=True,
    overwrite=True,
    gene_chunk_size=adf['gene_id'].n_unique(),
    sample_chunk_size=50_000,
)

In [ ]:
%%time

config_path = "/home/dnanexus/ukbgym/config_wgs_absplice2.yaml"
associations_df_path = '/home/dnanexus/data_dir/absplice2_assocs.parquet'
output_dir = "/home/dnanexus/debug_burdens_new"

adf = pl.read_parquet(associations_df_path).head(1)

get_burdens.compute_and_store_burdens(
    config_path=config_path,
    gene_list=adf['gene_id'].unique().to_list(),
    output_dir=output_dir,
    only_snps=True,
    na_mask=True,
    overwrite=True,
    gene_chunk_size=adf['gene_id'].n_unique(),
    sample_chunk_size=50_000,
)